In [2]:
from dataclasses import dataclass
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader
from minsearch import Index
from openai import OpenAI
from gitsource import chunk_documents
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# Initialize repository reader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
load_dotenv()

files = reader.read()
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

#Create chunk structure
chunks = chunk_documents(documents, size=2000, step=1000)
total_chunks = len(chunks)
print(f"Total number of chunks: {total_chunks}")

# Build search index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

# Templates
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["filename"])
        lines.append("content: " + doc["content"])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()
def search(query: str) -> dict[str, str]:
    """
    You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
    """
    return index.search(
        query,
        num_results=1,
        boost_dict={"question": 3.0, "content": 0.5},
    )
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

Total number of chunks: 295
-> Response received


-> Response received


-> Response received
